In [142]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn

In [143]:
df = pd.read_csv('../data/apartments_for_rent_classified_10K.csv', sep=';', encoding='cp1252')
df.head()

,id,category,title,body,amenities,bathrooms,bedrooms,currency,fee,has_photo,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,time
0,5668626895,housing/rent/apartment,"Studio apartment 2nd St NE, Uhland Terrace NE,...","This unit is located at second St NE, Uhland T...",NaN,NaN,0.0,USD,No,Thumbnail,...,$790,Monthly,101,NaN,Washington,DC,38.9057,-76.9861,RentLingo,1577359415
1,5664597177,housing/rent/apartment,Studio apartment 814 Schutte Road,"This unit is located at 814 Schutte Road, Evan...",NaN,NaN,1.0,USD,No,Thumbnail,...,$425,Monthly,106,814 Schutte Rd,Evansville,IN,37.9680,-87.6621,RentLingo,1577017063
2,5668626833,housing/rent/apartment,"Studio apartment N Scott St, 14th St N, Arling...","This unit is located at N Scott St, 14th St N,...",NaN,1.0,0.0,USD,No,Thumbnail,...,"$1,390",Monthly,107,NaN,Arlington,VA,38.8910,-77.0816,RentLingo,1577359410
3,5659918074,housing/rent/apartment,Studio apartment 1717 12th Ave,"This unit is located at 1717 12th Ave, Seattle...",NaN,1.0,0.0,USD,No,Thumbnail,...,$925,Monthly,116,1717 12th Avenue,Seattle,WA,47.6160,-122.3275,RentLingo,1576667743
4,5668626759,housing/rent/apartment,"Studio apartment Washington Blvd, N Cleveland ...","This unit is located at Washington Blvd, N Cle...",NaN,NaN,0.0,USD,No,Thumbnail,...,$880,Monthly,125,NaN,Arlington,VA,38.8738,-77.1055,RentLingo,1577359401


In [144]:
df.isnull().sum()

id                  0
category            0
title               0
body                0
amenities        3549
bathrooms          34
bedrooms            7
currency            0
fee                 0
has_photo           0
pets_allowed     4163
price               0
price_display       0
price_type          0
square_feet         0
address          3327
cityname           77
state              77
latitude           10
longitude          10
source              0
time                0
dtype: int64

In [145]:
# Drop the unwanted columns
df = df.drop(columns=['currency','fee', 'source', 'price_display'])

# Rename categories value to apartment and home 
df['category'] = df['category'].replace({
    'housing/rent/apartment': 'apartment',
    'housing/rent/home': 'home',
    'housing/rent/short_term': 'apartment'
})

df.head()

,id,category,title,body,amenities,bathrooms,bedrooms,has_photo,pets_allowed,price,price_type,square_feet,address,cityname,state,latitude,longitude,time
0,5668626895,apartment,"Studio apartment 2nd St NE, Uhland Terrace NE,...","This unit is located at second St NE, Uhland T...",NaN,NaN,0.0,Thumbnail,NaN,790,Monthly,101,NaN,Washington,DC,38.9057,-76.9861,1577359415
1,5664597177,apartment,Studio apartment 814 Schutte Road,"This unit is located at 814 Schutte Road, Evan...",NaN,NaN,1.0,Thumbnail,NaN,425,Monthly,106,814 Schutte Rd,Evansville,IN,37.9680,-87.6621,1577017063
2,5668626833,apartment,"Studio apartment N Scott St, 14th St N, Arling...","This unit is located at N Scott St, 14th St N,...",NaN,1.0,0.0,Thumbnail,NaN,1390,Monthly,107,NaN,Arlington,VA,38.8910,-77.0816,1577359410
3,5659918074,apartment,Studio apartment 1717 12th Ave,"This unit is located at 1717 12th Ave, Seattle...",NaN,1.0,0.0,Thumbnail,NaN,925,Monthly,116,1717 12th Avenue,Seattle,WA,47.6160,-122.3275,1576667743
4,5668626759,apartment,"Studio apartment Washington Blvd, N Cleveland ...","This unit is located at Washington Blvd, N Cle...",NaN,NaN,0.0,Thumbnail,NaN,880,Monthly,125,NaN,Arlington,VA,38.8738,-77.1055,1577359401


In [146]:
df['bedrooms'] = (
    pd.to_numeric(df['bedrooms'], errors='coerce')
    .fillna(0)
    .astype(int)
)

In [147]:
df.isnull().sum()

id                 0
category           0
title              0
body               0
amenities       3549
bathrooms         34
bedrooms           0
has_photo          0
pets_allowed    4163
price              0
price_type         0
square_feet        0
address         3327
cityname          77
state             77
latitude          10
longitude         10
time               0
dtype: int64

In [148]:
df['amenities'] = (
    df['amenities']
    .fillna('')
    .str.replace(r'\bLuxury\b,?\s*', '', regex=True)
    .str.replace(r',\s*,', ',', regex=True)   # fix double commas
    .str.strip(', ')                          # clean edges
)

In [152]:
# Assuming 'rent' is your DataFrame, as per previous interactions.


# 1. Define the complete list of unique amenity keywords from your provided data.
# This list is derived directly from the amenities table.
amenity_keywords = [
    'Parking', 'Dishwasher', 'Pool', 'Refrigerator', 'Patio/Deck',
    'Cable or Satellite', 'Storage', 'Gym', 'Internet Access', 'Clubhouse',
    'Garbage Disposal', 'Washer Dryer', 'Fireplace', 'Playground', 'AC',
    'Elevator', 'Tennis', 'Gated', 'Wood Floors', 'Hot Tub', 'Wifi', 'Internet', 
    'Basketball', 'TV', 'View', 'Doorman', 'Alarm', 'Golf', 'Garage', 'Amenities', 
    'Amenity', 'Laundry', 'Laundry Room', 'Laundry Facility', 'Laundry Onsite', 'Laundry In Unit',
    'Playroom', 'Study Room', 'Spa', 'Microwave', 'Oven', 'BBQ', 'Patio', 'Deck'
]


# 2. Iterate through each row in the DataFrame to perform the check and update.
for index, row in df.iterrows():
    # Get the body text and amenities, handling potential missing values.
    # Convert body to string and lowercase for case-insensitive matching.
    body_text = str(row['body']).lower() if pd.notna(row['body']) else ""
   
    # Handle the existing amenities column. We'll split it into a list for easy checking.
    # If the column is NaN, we treat it as an empty list.
    if pd.isna(row['amenities']):
        existing_amenities_list = []
    else:
        # Split by comma to get individual amenities. We can assume the input might be like the examples.
        # Converting existing to lowercase for robust comparison.
        existing_amenities_list = [a.strip().lower() for a in str(row['amenities']).split(',')]


    # 3. Identify keywords in 'body' that are NOT in the 'amenities' list.
    missing_amenities = []
    for keyword in amenity_keywords:
        keyword_lower = keyword.lower()
        # Check if the lowercased keyword is in the lowercased body text.
        if keyword_lower in body_text:
            # Check if this keyword is NOT already present in the existing amenities list.
            if keyword_lower not in existing_amenities_list:
                # Add the original-cased keyword to our list of missing ones.
                missing_amenities.append(keyword)


    # 4. If any missing amenities were found, update the 'amenities' column for that row.
    if missing_amenities:
        # Create a string of the missing amenities, joined by a comma and space.
        new_amenities_string = ', '.join(missing_amenities)
       
        # Determine the final updated amenities string.
        # If the original was NaN (no initial amenities), just use the new ones.
        if pd.isna(row['amenities']):
            updated_amenities = new_amenities_string
        else:
            # Append the new string to the existing, separated by a comma.
            # This replicates the format shown in the example images.
            updated_amenities = f"{row['amenities']}, {new_amenities_string}"
       
        # Update the cell directly in the DataFrame.
        df.at[index, 'amenities'] = updated_amenities


# Optional: Show the first few rows of the updated columns to verify.
print("Successfully checked 'body' for missing keywords and updated 'amenities' column.")
display(df[['body', 'amenities']].head())


Successfully checked 'body' for missing keywords and updated 'amenities' column.


,body,amenities
0,"This unit is located at second St NE, Uhland T...",", AC"
1,"This unit is located at 814 Schutte Road, Evan...",
2,"This unit is located at N Scott St, 14th St N,...",
3,"This unit is located at 1717 12th Ave, Seattle...",
4,"This unit is located at Washington Blvd, N Cle...",


In [153]:
df['has_amenities'] = np.where(
    df['amenities'].notna() & (df['amenities'].str.strip() != ''),
    'yes',
    'no'
)
df.head()

,id,category,title,body,amenities,bathrooms,bedrooms,has_photo,pets_allowed,price,price_type,square_feet,address,cityname,state,latitude,longitude,time,has_amenities
0,5668626895,apartment,"Studio apartment 2nd St NE, Uhland Terrace NE,...","This unit is located at second St NE, Uhland T...",", AC",NaN,0,Thumbnail,NaN,790,Monthly,101,NaN,Washington,DC,38.9057,-76.9861,1577359415,yes
1,5664597177,apartment,Studio apartment 814 Schutte Road,"This unit is located at 814 Schutte Road, Evan...",,NaN,1,Thumbnail,NaN,425,Monthly,106,814 Schutte Rd,Evansville,IN,37.9680,-87.6621,1577017063,no
2,5668626833,apartment,"Studio apartment N Scott St, 14th St N, Arling...","This unit is located at N Scott St, 14th St N,...",,1.0,0,Thumbnail,NaN,1390,Monthly,107,NaN,Arlington,VA,38.8910,-77.0816,1577359410,no
3,5659918074,apartment,Studio apartment 1717 12th Ave,"This unit is located at 1717 12th Ave, Seattle...",,1.0,0,Thumbnail,NaN,925,Monthly,116,1717 12th Avenue,Seattle,WA,47.6160,-122.3275,1576667743,no
4,5668626759,apartment,"Studio apartment Washington Blvd, N Cleveland ...","This unit is located at Washington Blvd, N Cle...",,NaN,0,Thumbnail,NaN,880,Monthly,125,NaN,Arlington,VA,38.8738,-77.1055,1577359401,no


In [154]:
df['has_amenities'].value_counts()

has_amenities
yes    7082
no     2918
Name: count, dtype: int64